# 演習1 解答編 ―― スレッドとパイプライン

> まず `ex01_threads.ipynb` を自分で解いてから読んでください。
> このノートには、答えを**自分で確かめるためのコード**も入っています。上から順に ▶ を押してください。

## 発展課題1、2 の解答 ―― ボトルネックは動く

次のセルを実行すると、3つの場合について **3フレーム分**の逐次／パイプラインの所要時間と、
ボトルネックがどの段かが出ます。


In [ ]:
%%writefile ans01a.cpp
#include <iostream>
#include <iomanip>
#include <algorithm>
using namespace std;
const char* SN[3] = {"Read", "Infer", "Show"};

// n フレーム処理し終わるまでの時間(ms)を返す
int finish(int t[3], int n, bool pipeline) {
    int fin[3] = {0, 0, 0}, last = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < 3; j++) {
            int wait = pipeline ? fin[j] : last;
            int s = max(j ? fin[j - 1] : 0, wait);
            fin[j] = s + t[j];
            if (j == 2) last = fin[2];
        }
    return last;
}

void show(const char* label, int t[3], int n) {
    int a = finish(t, n, false), b = finish(t, n, true);
    int bn = 0;
    for (int j = 1; j < 3; j++) if (t[j] > t[bn]) bn = j;
    cout << left << setw(24) << label << right << fixed << setprecision(1)
         << setw(6) << a << "ms" << setw(6) << 1000.0 * n / a << "FPS"
         << setw(8) << b << "ms" << setw(6) << 1000.0 * n / b << "FPS"
         << "    " << SN[bn] << "(" << t[bn] << "ms)"
         << " 上限" << setprecision(0) << 1000.0 / t[bn] << "FPS\n";
}

int main() {
    int a[3] = {30, 30, 30};
    int b[3] = {13, 67, 33};      // 真ん中が重いとき
    int c[3] = {13,  2, 33};      // 真ん中だけを大幅に速くしたとき
    cout << "3フレーム分             逐次            パイプライン       ボトルネック\n";
    cout << "-------------------------------------------------------------------------\n";
    show("30 / 30 / 30", a, 3);
    show("13 / 67 / 33", b, 3);
    show("13 /  2 / 33", c, 3);
    return 0;
}

In [ ]:
!g++ -std=c++17 ans01a.cpp -o ans01a && ./ans01a

**1. `{13, 67, 33}` ―― 真ん中が重い**

3フレーム分で、逐次 339ms（8.8 FPS）→ パイプライン 247ms（12.1 FPS）。約1.4倍です。
図にすると Infer の行だけがびっしり埋まり、**Read と Show の行は `.` だらけ**になります。
十分に長く流し続けたときの上限は `1000 / 67 ≒ 15 FPS`。ここが天井です。

**2. `{13, 2, 33}` ―― 真ん中だけを大幅に速くした**

3フレーム分で、逐次 144ms（20.8 FPS）→ パイプライン 114ms（26.3 FPS）。
**ボトルネックは真ん中から Show へ移ります。**
上限は Show の 33ms で決まり、`1000 / 33 ≒ 30 FPS`。
真ん中を 67ms → 2ms と **33倍速く**したのに、上限は 15 FPS → 30 FPS の **2倍**にしかなりません。

> **一番遅い段を速くしない限り、全体は速くならない。**

これは **アムダールの法則** と呼ばれる考え方の、いちばん分かりやすい形です。
「速くした部分の効果は、それが全体に占める割合までしか出ない」。

なお、3フレームのような短い区間では、パイプラインの**立ち上がり**（全部の段が埋まるまで）と
**終わり際**が効くので、FPS は上限より低く出ます。フレーム数を増やすほど上限に近づきます。

そして、この状況で **真ん中の担当を増やすことにはまったく意味がありません**（発展課題3の答え）。
すでに手待ちの係を増やしても、詰まっている Show は1ミリも速くなりません。


## 発展課題3、4 の解答 ―― 人を増やしても倍にならないとき

発展課題3の答えは上で見たとおり「意味がない」です。ボトルネックが別の段にあるからです。

では発展課題4、**ボトルネックの段そのものを2人に増やしたら、必ず倍になる**のでしょうか。
なりません。代表的な理由が2つあります。

### 理由① コアが足りない

1-2 で見たとおりです。計算が中心の仕事（CPUバウンド）は、
「同時に計算できる数」で頭打ちになります。

### 理由② その仕事が「1つしかないもの」を使っている

こちらのほうが見落とされがちです。

2人の担当者が、途中で**同じ1つの資源**を使わなければならない場合、
そこは**一度に1人しか通れません**。たとえば、

- **1台しかない外部ハードウェア**（GPU、アクセラレータ、プリンタ、計測器）
- 1本しかない通信路やファイル
- 排他制御された共有データ

こういう仕事の中身は、たいてい次のように分かれています。

```
[ 準備(CPU) ] → [ 共有資源を使う ] → [ 後始末(CPU) ]
                  ↑ ここだけは順番待ち
```

準備と後始末は2人が同時にできますが、真ん中は必ず順番待ちです。

### タイムラインで見る

**準備30ms → 共有資源60ms → 後始末10ms** という仕事を3件やってみます。図の読み方は 1-3 と同じで、

- 横が時間。**1文字 = 5ms**、数字は**何件目の仕事か**
- `P` = 準備（30ms、同時にできる）、`H` = 共有資源を使う（60ms、一度に1人）、`Q` = 後始末（10ms）
- `HW` の行 = **共有資源が誰に使われているか**
- `.` = 手待ち（`HW` の行では「誰も使っていない」）

```
【1人でやる】
W1    P11111H11111111111Q1P22222H22222222222Q2P33333H33333333333Q3
HW    ......111111111111........222222222222........333333333333..
Total 300 ms
```

`HW` の行を見てください。W1 が `P`（準備）や `Q`（後始末）をしている間、
**共有資源は誰にも使われずに遊んでいます**。
3件で 300ms かかったのに、資源が実際に働いたのは 180ms 分だけです。

```
【2人でやる】   W1 が1・3件目、W2 が2件目を担当
W1    P11111H11111111111Q1P33333....H33333333333Q3
W2    P22222............H22222222222Q2............
HW    ......111111111111222222222222333333333333..
Total 220 ms
```

**`HW` の行が隙間なく埋まりました。** 30ms から 210ms までずっと使われています。
W1 が準備・後始末をしている裏で W2 が資源を使い、その逆も起きているからです。

その代わり、**W1・W2 の行には `.`（順番待ち）が現れます。**
W1 は 130ms に共有資源を使いたいのに、W2 が使い終わる 150ms まで待たされています。
結果は 300ms → 220ms。**2人にしても倍にはなりません。**

理論下限は「共有資源の使用時間の合計」＝ 60ms × 3件 = **180ms**。
人を何人に増やしても、これより速くなることはありません。

つまり **2人にする目的は「2倍速くする」ことではなく「共有資源を遊ばせないこと」**。
上限は `1 / 共有資源の使用時間` で頭打ちになります。

次のセルで確かめてください。同じ仕事を 1人 / 2人 / 4人、12件でやらせます。


In [ ]:
%%writefile ans01b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::mutex shared_hw;      // 1つしかない資源 = 一度に1スレッドしか使えない

void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void worker(int id, int nworker, int njob) {
    for (int f = id; f < njob; f += nworker) {
        wait_ms(30);                                                 // 準備（CPU）
        { std::lock_guard<std::mutex> g(shared_hw); wait_ms(60); }   // 共有資源（順番待ち）
        wait_ms(10);                                                 // 後始末（CPU）
    }
}

void run(int nworker, int njob) {
    auto t0 = steady_clock::now();
    std::vector<std::thread> ts;
    for (int k = 0; k < nworker; k++) ts.emplace_back(worker, k, nworker, njob);
    for (auto& t : ts) t.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();
    std::cout << nworker << "人 : " << ms << " ms  ->  "
              << (1000.0 * njob / ms) << " 件/秒\n";
}

int main() {
    std::cout << "準備30ms + 共有資源60ms(一度に1人) + 後始末10ms、12件\n\n";
    run(1, 12);
    run(2, 12);
    run(4, 12);
    std::cout << "\n共有資源だけの理論下限 = 60ms x 12 = 720 ms -> 16.7 件/秒\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans01b.cpp -o ans01b && ./ans01b

1人 → 2人では大きく速くなり（共有資源の空き時間が埋まる）、
**2人 → 4人ではまったく変わりません**。理論下限に張り付いているからです。

> **`std::mutex`（`lock_guard`）は演習3で扱います。**
> ここでは「1つしかない資源＝一度に1人しか使えない」を表す道具として使っているだけです。
> いまは意味が分からなくても構いません。

まとめると、担当者を増やす前に確かめるべきことは2つです。

1. **その段は本当にボトルネックか**（別の段が詰まっていないか）
2. **その段は本当に並列に動けるか**（コアは足りるか、1つしかない資源を取り合っていないか）

どちらも「増やしてから測る」のではなく、**増やす前に考えられる**ことです。

## 発展課題5 の解答 ―― `join()` を消すと

**プログラムが異常終了します**（`terminate called without an active exception`）。

In [ ]:
%%writefile ans01c.cpp
#include <iostream>
#include <thread>
#include <chrono>

void work() {
    std::this_thread::sleep_for(std::chrono::milliseconds(100));
    std::cout << "スレッドの仕事が終わった\n";
}

int main() {
    std::thread t(work);
    // t.join();          // ← これを消すと？
    std::cout << "main が終わる\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans01c.cpp -o ans01c
!./ans01c || echo "→ 異常終了しました"

`std::thread` のオブジェクトは、**まだ走っているスレッドを抱えたまま破棄されると強制終了する**
決まりになっています。「放置されたスレッドがある」状態を許さない、という設計です。

`join()` は「そのスレッドが終わるまで待って、後片付けをする」処理です。
スレッドを起動したら、必ずどこかで `join()`（または `detach()`）する必要があります。

スレッドを複数まとめて起動したときは、全部について `join()` します。

```cpp
std::vector<std::thread> ts;
...
for (auto& t : ts) t.join();      // 起動した全員を待つ
```

> **`detach()` という選択肢もあります**が、こちらは「もう面倒を見ない」という宣言で、
> そのスレッドが終わったかどうかを知る手段がなくなります。
> 「起動したら `join()`」を基本にしてください。
> スレッドをどう安全に終わらせるかは、演習9で改めて扱います。

## 発展課題6 の解答 ―― コアが2つなのにスレッドを3本立てたら

### (a) 3本目も動きます

**「コアが2つだから2本しか動かない」ではありません。** 3本とも動きます。
OS が非常に短い間隔で担当を入れ替えるからです（**タイムスライス**と呼びます）。

ある瞬間を切り取れば、走っているのは確かに2本だけです。
しかし切り取る瞬間を変えると、走っている2本の顔ぶれが変わっています。

### (b) 全部で約 450ms。3本はほぼ同時に終わります

図にします。**1文字 = 50ms**、`A` `B` `C` はそれぞれ T1・T2・T3 が走っている時間、
`.` は「順番待ちで止められている」時間です。
（実際の入れ替わりは数ms単位でずっと細かいので、これは模式図です。）

```
【まずコア2つ・スレッド2本】   1本あたり300msの計算
T1    AAAAAA
T2    BBBBBB
Core  222222      ← どの瞬間も2つのコアが埋まっている
Total 300 ms
```

```
【コア2つ・スレッド3本】   1本あたり300msの計算
T1    AA.AA.AA.
T2    B.BB.BB.B
T3    .CC.CC.CC
Core  222222222   ← やはりどの瞬間も2つ。コアは1つも遊んでいない
Total 450 ms
```

- どの列も `A` `B` `C` のうち**ちょうど2つ**が立っています。コアは常に満杯です
- 各スレッドは自分の行に `A` を6文字（＝300ms分）持っています。ちゃんと全員が仕事を終えています
- ただし、**実時間のうち自分が走れるのは 2/3 だけ**。だから 300ms の仕事に 450ms かかります
- そして3本は**ほぼ同時に**終わります。1本ずつ順に終わるのではありません

式にすると単純です。

```
全部終わるまでの時間 ≒ 仕事の総量 ÷ コア数
                    = 300ms × 3本 ÷ 2コア
                    = 450ms
```

スレッドを4本、5本と増やしても、**総量 ÷ コア数** は変わりません。
計算する仕事では、スレッドを増やしても**総時間は減らない**のです。
減らないどころか、入れ替えの手間（**コンテキストスイッチ**）の分だけ少し損をします。

> **Colab で試すときの注意**：1-2 で見たとおり、Colab は `hardware_concurrency()` が 2 でも
> 実際の計算能力は1コア分しかないことがよくあります。その場合は上の式の「コア数」を **1** として
> 当てはめてください（3本で約900ms）。式そのものは変わりません。

### (c) 待つだけの仕事なら、答えは変わります

```
【コア2つ・スレッド3本】   1本あたり300ms「待つだけ」
T1    AAAAAA
T2    BBBBBB
T3    CCCCCC
Core  000000      ← 待っている間、コアは使っていない
Total 300 ms
```

**3本とも 300ms で終わります。** `.`（順番待ち）が1つも出てきません。
待つのに計算回路は要らないので、コア数はまったく関係ないからです。
10本立てても、100本立てても 300ms です。

### なぜこれが大事なのか

パイプラインの各段は、たいてい「**頼んで、待つ**」仕事です。

- ファイルやカメラから読む → デバイスに頼んで、届くのを待つ
- 外部ハードウェア（DPU など）に推論させる → 頼んで、結果を待つ
- 画面に出す → 描画に頼んで、終わるのを待つ

だから **コアが2つしかないマシンでも、段を3つ4つに分けたパイプラインはちゃんと速くなります。**
1-3 の演習が2コアの Colab で成立するのは、このためです。

逆に、各段が「ひたすら計算する」仕事だった場合は、
段をいくら増やしてもコア数の壁を越えられません。

> **まとめ**：スレッドを何本立ててよいかを決めるのは、コア数ではなく
> **その仕事が「計算する仕事」か「待つ仕事」か**です。



---

## 参考：本番のプログラムでは

ハッカソンで読む `yolov3_video_study.cpp` は、まさにこの形をしています。

- スレッドは4本（読み込み1・推論2・表示1）で、キューでつないだ**パイプライン**
- 推論スレッドの中身は「前処理(CPU) → DPU推論 → 後処理(CPU)」
- **DPU はボード上に1個しかない**ので、推論2本は DPU の前で順番待ちになる
  （＝発展課題4の「1つしかない資源」そのもの）
- `main` の冒頭で `hardware_concurrency()` を表示している

いま学んだことが、そのまま「なぜこの構成なのか」の説明になります。
細かい読み方は演習を進めながら扱うので、いまは眺めるだけで構いません。